In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga del shapefile de secciones censales

In [2]:
# Cargo el GDF con las secciones censales
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)

# Compruebo que se haya cargado
print(f"Vista del gdf de secciones:\n{gdf_secciones.head()}")

Vista del gdf de secciones:
        CUSEC  CUMUN CSEC CDIS CMUN CPRO CCA    CUDIS  CLAU2   NPRO  \
0  0500101001  05001  001   01  001   05  07  0500101  05001  Ávila   
1  0500201001  05002  001   01  002   05  07  0500201  05002  Ávila   
2  0500201002  05002  002   01  002   05  07  0500201  05002  Ávila   
3  0500501001  05005  001   01  005   05  07  0500501  05005  Ávila   
4  0500701001  05007  001   01  007   05  07  0500701  05007  Ávila   

               NCA CNUT0 CNUT1 CNUT2 CNUT3                      NMUN  \
0  Castilla y León    ES     4     1     1                   Adanero   
1  Castilla y León    ES     4     1     1                Adrada, La   
2  Castilla y León    ES     4     1     1                Adrada, La   
3  Castilla y León    ES     4     1     1                  Albornos   
4  Castilla y León    ES     4     1     1  Aldeanueva de Santa Cruz   

                                            geometry  
0  POLYGON ((365705.918 4536187.034, 365958.915 4...  
1 

# Carga del fichero de centros docentes

El fichero de centros docentes se extrae de datosabiertos.jcyl disponible en la siguiente dirección: 

https://datosabiertos.jcyl.es/web/jcyl/set/es/educacion/centrosdocentes/1284200521439

In [3]:
# Cargo el fichero csv con los centros docentes
ruta_csv_centros_docentes = os.path.join(DATA_INPUTS_DA, "directorio-de-centros-docentes.csv")
df_centros_docentes = pd.read_csv(ruta_csv_centros_docentes, sep=";", encoding="utf-8")
df_centros_docentes["centro_id"] = range(1, len(df_centros_docentes) + 1)

# Veo una muestra de su estructura y contenido
print(df_centros_docentes.info())
df_centros_docentes.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1754 entries, 0 to 1753
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CURSO ACADÉMICO              1754 non-null   int64  
 1   CÓDIGO                       1754 non-null   int64  
 2   C.SITUACIÓN                  1754 non-null   object 
 3   SITUACIÓN                    1754 non-null   object 
 4   C.NATURALEZA                 1754 non-null   int64  
 5   NATURALEZA                   1754 non-null   object 
 6   C.DENOMINACIÓN GENÉRICA      1754 non-null   int64  
 7   DENOMINACIÓN GENÉRICA        1754 non-null   object 
 8   DENOMINACIÓN GENÉRICA BREVE  1754 non-null   object 
 9   DENOMINACIÓN ESPECÍFICA      1754 non-null   object 
 10  C.VÍA                        1754 non-null   object 
 11  VÍA                          1754 non-null   object 
 12  NOMBRE DE LA VÍA             1754 non-null   object 
 13  NÚMERO            

,CURSO ACADÉMICO,CÓDIGO,C.SITUACIÓN,SITUACIÓN,C.NATURALEZA,NATURALEZA,C.DENOMINACIÓN GENÉRICA,DENOMINACIÓN GENÉRICA,DENOMINACIÓN GENÉRICA BREVE,DENOMINACIÓN ESPECÍFICA,...,COORD. LONGITUD,COORD. LATITUD,C.R.A,INTERNADO,CONCIERTO,JORNADA CONTINUA,COMEDOR,TRANSPORTE,Localización,centro_id
1186,2025,49010965,A,ALTA,2,PRIVADO,4,CENTRO PRIVADO DE EDUCACION INFANTIL,CPrEI,DUENDE,...,-5.74075,41.51275,N,N,N,N,N,N,"41.51275, -5.74075",1187
1730,2025,24014150,A,ALTA,1,PÚBLICO,42,INSTITUTO DE EDUCACION SECUNDARIA,IES,OBISPO ARGÜELLES,...,-6.32298,42.93638,N,N,N,S,N,S,"42.93638, -6.32298",1731
997,2025,24022298,A,ALTA,1,PÚBLICO,3,ESCUELA DE EDUCACION INFANTIL,EEI,PEQUECOYANZA,...,-5.51491,42.28873,N,N,N,N,S,N,"42.28873, -5.51491",998
157,2025,34003971,A,ALTA,1,PÚBLICO,3,ESCUELA DE EDUCACION INFANTIL,EEI,VIRGEN DEL CARMEN,...,-4.52742,42.00512,N,N,N,N,S,N,"42.00512, -4.52742",158
943,2025,47011361,A,ALTA,1,PÚBLICO,3,ESCUELA DE EDUCACION INFANTIL,EEI,BABYLANDIA - LA CASA DE LOS NIÑOS,...,-4.58797,41.34016,N,N,N,N,S,N,"41.34016, -4.58797",944


# Filtrado del fichero

## Naturaleza
Como se va a analizar la vulnerabilidad socio-territorial, se van a filtrar todos aquellos centros cuya naturaleza pertenezca al ámbito privado

## Denominación genérica
Del mismo modo, se excluyen del análisis aquellos centros que no den cobertura a las necesitades básicas (por ejemplo conservatorios, escuelas de
arte...). Además, se agrupan en un número menor de categorías para facilitar el análisis posterior. LAs categorías finales que resultarán serán:

1. Infantil y primaria
2. Secundaria y bachillerato
3. Formación profesional
4. Educación especial
5. Adultos e idiomas
6. Artes y música
7. Otros

In [4]:
# Filtro por los centros docenets de C_NATURALEZA = 1 (Publicos)
df_centros_docentes_publicos = df_centros_docentes[df_centros_docentes["NATURALEZA"] == "PÚBLICO"].copy()

# Reviso las distintas denominaciones genéricas
print(f"Recuento de centro disponibles por denominación:\n\n{df_centros_docentes_publicos['DENOMINACIÓN GENÉRICA'].value_counts()}\n")

# Creo unas categorías de centro simplificadas

mapa_denominaciones = {
    # Infantil y Primaria
    'ESCUELA DE EDUCACION INFANTIL': 'INFANTIL Y PRIMARIA',
    'COLEGIO DE EDUCACION INFANTIL Y PRIMARIA': 'INFANTIL Y PRIMARIA',
    'COLEGIO DE EDUCACION PRIMARIA': 'INFANTIL Y PRIMARIA',

    # Secundaria y Bachillerato
    'INSTITUTO DE EDUCACION SECUNDARIA': 'SECUNDARIA Y BACHILLERATO',
    'INSTITUTO DE EDUCACION SECUNDARIA OBLIGATORIA': 'SECUNDARIA Y BACHILLERATO',
    'CENTRO DE EDUCACIÓN OBLIGATORIA': 'SECUNDARIA Y BACHILLERATO',

    # Formación Profesional
    'CENTRO PUBLICO INTEGRADO DE FORMACION PROFESIONAL': 'FORMACIÓN PROFESIONAL',

    # Educación Especial
    'COLEGIO DE EDUCACION ESPECIAL': 'EDUCACIÓN ESPECIAL',

    # Adultos e Idiomas
    'CENTRO PUBLICO DE EDUCACION DE PERSONAS ADULTAS': 'ADULTOS E IDIOMAS',
    'ESCUELA OFICIAL DE IDIOMAS': 'ADULTOS E IDIOMAS',

    # Artes y Música
    'ESCUELA DE ARTE Y SUPERIOR DE DISEÑO': 'ARTES Y MÚSICA',
    'ESCUELA DE ARTE Y SUPERIOR DE DISEÑO Y DE CONS. Y RESTAUR. DE BIENES CULTURALES': 'ARTES Y MÚSICA',
    'ESCUELA DE ARTE Y SUPERIOR DE CONSERVACIÓN Y RESTAURACIÓN DE BIENES CULTURALES': 'ARTES Y MÚSICA',
    'ESCUELA DE DANZA': 'ARTES Y MÚSICA',
    'ESCUELA DE MUSICA': 'ARTES Y MÚSICA',
    'ESCUELA DE MUSICA Y DANZA': 'ARTES Y MÚSICA',
    'CONSERVATORIO ELEMENTAL DE MUSICA': 'ARTES Y MÚSICA',
    'CONSERVATORIO PROFESIONAL DE MUSICA': 'ARTES Y MÚSICA',
    'CONSERVATORIO SUPERIOR DE MUSICA': 'ARTES Y MÚSICA',

    # Otros (sin impacto directo en accesibilidad social)
    'CENTRO DOCENTE DE FORMACIÓN MILITAR': 'OTROS'
}

# Asignación de categoría agregada
df_centros_docentes_publicos["CATEGORIA"] = df_centros_docentes_publicos[
    "DENOMINACIÓN GENÉRICA"
].map(mapa_denominaciones)

# Visualización de conteos por categoría
print(f"Recuento de centro disponibles por categoría:\n\n{df_centros_docentes_publicos['CATEGORIA'].value_counts()}")

# Selección de categorías relevantes para análisis de vulnerabilidad
categorias_relevantes = [
    "INFANTIL Y PRIMARIA",
    "SECUNDARIA Y BACHILLERATO",
    "FORMACIÓN PROFESIONAL",
    "EDUCACIÓN ESPECIAL",
    "ADULTOS E IDIOMAS"
]

# Filtro final
df_centros_docentes_relevantes = df_centros_docentes_publicos[
    df_centros_docentes_publicos["CATEGORIA"].isin(categorias_relevantes)
].copy()

# Muestra
df_centros_docentes_relevantes.sample(5)

Recuento de centro disponibles por denominación:

DENOMINACIÓN GENÉRICA
COLEGIO DE EDUCACION INFANTIL Y PRIMARIA                                           624
ESCUELA DE EDUCACION INFANTIL                                                      215
INSTITUTO DE EDUCACION SECUNDARIA                                                  191
ESCUELA DE MUSICA                                                                   68
CENTRO PUBLICO DE EDUCACION DE PERSONAS ADULTAS                                     54
CENTRO PUBLICO INTEGRADO DE FORMACION PROFESIONAL                                   28
INSTITUTO DE EDUCACION SECUNDARIA OBLIGATORIA                                       19
ESCUELA OFICIAL DE IDIOMAS                                                          14
COLEGIO DE EDUCACION ESPECIAL                                                       12
CENTRO DE EDUCACIÓN OBLIGATORIA                                                     11
CONSERVATORIO PROFESIONAL DE MUSICA                       

,CURSO ACADÉMICO,CÓDIGO,C.SITUACIÓN,SITUACIÓN,C.NATURALEZA,NATURALEZA,C.DENOMINACIÓN GENÉRICA,DENOMINACIÓN GENÉRICA,DENOMINACIÓN GENÉRICA BREVE,DENOMINACIÓN ESPECÍFICA,...,COORD. LATITUD,C.R.A,INTERNADO,CONCIERTO,JORNADA CONTINUA,COMEDOR,TRANSPORTE,Localización,centro_id,CATEGORIA
1094,2025,34001901,A,ALTA,1,PÚBLICO,42,INSTITUTO DE EDUCACION SECUNDARIA,IES,VIRGEN DE LA CALLE,...,42.02132,N,N,N,S,N,S,"42.02132, -4.52963",1095,SECUNDARIA Y BACHILLERATO
767,2025,24016559,A,ALTA,1,PÚBLICO,31,CENTRO PUBLICO DE EDUCACION DE PERSONAS ADULTAS,CEPA,RAMÓN CARNICER,...,42.55204,N,N,N,N,N,N,"42.55204, -6.59795",768,ADULTOS E IDIOMAS
1033,2025,9008548,A,ALTA,1,PÚBLICO,42,INSTITUTO DE EDUCACION SECUNDARIA,IES,ALFOZ DE LARA,...,42.01761,N,N,N,S,N,S,"42.01761, -3.27435",1034,SECUNDARIA Y BACHILLERATO
351,2025,47003246,A,ALTA,1,PÚBLICO,14,COLEGIO DE EDUCACION INFANTIL Y PRIMARIA,CEIP,GABRIEL Y GALÁN,...,41.65304,N,N,N,S,S,S,"41.65304, -4.71135",352,INFANTIL Y PRIMARIA
1103,2025,47003519,A,ALTA,1,PÚBLICO,14,COLEGIO DE EDUCACION INFANTIL Y PRIMARIA,CEIP,ANTONIO ALLÚE MORER,...,41.63131,N,N,N,S,S,N,"41.63131, -4.72466",1104,INFANTIL Y PRIMARIA


# Asignación de código de sección censal (CUSEC)
Como no se dispone por defecto del CUSEC pero sí de las coordenadas geográficas del centro, y se dispone de un shapefile con los
códigos de sección de Castilla y León georreferenciados, puede asignarse el CUSEC a este conjunto de datos mediante un join espacial

In [5]:
# Utilizo una función definida para integrar el cusec
df_centros_docentes_relevantes = asignar_cusec_por_coordenadas(
    df_centros_docentes_relevantes, "COORD. LONGITUD", "COORD. LATITUD")

# Compruebo que tiene ahora la columna cusec
df_centros_docentes_relevantes.sample(5)

,CURSO ACADÉMICO,CÓDIGO,C.SITUACIÓN,SITUACIÓN,C.NATURALEZA,NATURALEZA,C.DENOMINACIÓN GENÉRICA,DENOMINACIÓN GENÉRICA,DENOMINACIÓN GENÉRICA BREVE,DENOMINACIÓN ESPECÍFICA,...,C.R.A,INTERNADO,CONCIERTO,JORNADA CONTINUA,COMEDOR,TRANSPORTE,Localización,centro_id,CATEGORIA,CUSEC
230,2025,42003761,A,ALTA,1,PÚBLICO,42,INSTITUTO DE EDUCACION SECUNDARIA,IES,SANTA CATALINA,...,N,N,N,S,N,S,"41.58896, -3.06155",231,SECUNDARIA Y BACHILLERATO,4204301001
1475,2025,49006706,A,ALTA,1,PÚBLICO,14,COLEGIO DE EDUCACION INFANTIL Y PRIMARIA,CEIP,MIGUEL DE CERVANTES,...,N,N,N,S,S,N,"41.50956, -5.73083",1476,INFANTIL Y PRIMARIA,4927502010
1719,2025,47006673,A,ALTA,1,PÚBLICO,42,INSTITUTO DE EDUCACION SECUNDARIA,IES,JUAN DE JUNI,...,N,N,N,S,N,S,"41.6603, -4.72883",1720,SECUNDARIA Y BACHILLERATO,4718608024
467,2025,40003411,A,ALTA,1,PÚBLICO,14,COLEGIO DE EDUCACION INFANTIL Y PRIMARIA,CEIP,VILLALPANDO,...,N,N,N,S,S,N,"40.93928, -4.11341",468,INFANTIL Y PRIMARIA,4019404005
121,2025,42007365,A,ALTA,1,PÚBLICO,3,ESCUELA DE EDUCACION INFANTIL,EEI,LANGA,...,N,N,N,N,N,N,"41.7666, -2.47903",122,INFANTIL Y PRIMARIA,4217302007


In [6]:
# Miro a ver si algun centro queda sin CUSEC
print(f"Centros sin CUSEC asignado: {df_centros_docentes_relevantes['CUSEC'].isna().sum()}")

# Miro a ver si tienen codigos postales
print(f"Codigos postales:\n{df_centros_docentes_relevantes[df_centros_docentes_relevantes['CUSEC'].isna()]['C.POSTAL']}")

# Como está la columna C.POSTAL llamo a la función para obtener el CUSEC a partir del CP
df_centros_docentes_relevantes = asignar_cusec_por_cp(df_centros_docentes_relevantes, columna_cp = "C.POSTAL")

Centros sin CUSEC asignado: 4
Codigos postales:
1263    37339
1275     9197
1288    40140
1292     9314
Name: C.POSTAL, dtype: int64
✅ 1169 registros procesados | 🔁 4 CUSEC completados | ❗ 0 sin asignar


# Cálculo de distancia mínima a un servicio desde un CUSEC y accesibilidad de servicios para un CUSEC
Es necesario para cada sección censal calcular la distancia mínima existente hacia un servicio de un tipo, así como crear
una medida de accesibilidad de un servicio concreto desde una sección cens

In [10]:
# Resultado final por CUSEC
df_resultado = pd.DataFrame({"CUSEC": gdf_secciones["CUSEC"].unique()})

# Tabla global de relaciones
lista_relaciones = []

for categoria in df_centros_docentes_relevantes["CATEGORIA"].dropna().unique():
    print(f"\n➡️ Procesando categoría: {categoria}")

    # Filtrar la categoría
    df_cat = df_centros_docentes_relevantes[
        df_centros_docentes_relevantes["CATEGORIA"] == categoria
    ].copy()

    # Normalizar nombre para usarlo como prefijo de columnas
    nombre_norm = (
        categoria.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )

    # Calcular accesibilidad (versión v2 o v3)
    df_acc, df_rel = calcular_accesibilidad_v2(
        df_servicio=df_cat,
        nombre_servicio=nombre_norm,
        col_id="centro_id"              # ID del centro educativo
    )

    # Acumular relaciones
    lista_relaciones.append(df_rel)

    # Unir métricas agregadas al DF principal
    df_resultado = df_resultado.merge(df_acc, on="CUSEC", how="outer")

# Concatenar todas las relaciones
df_relaciones_global = pd.concat(lista_relaciones, ignore_index=True)

df_resultado.head()


➡️ Procesando categoría: INFANTIL Y PRIMARIA


Accesibilidad infantil_y_primaria: 100%|█████████████████████████████████████| 3535/3535 [00:17<00:00, 197.62sección/s]



➡️ Procesando categoría: ADULTOS E IDIOMAS


Accesibilidad adultos_e_idiomas: 100%|███████████████████████████████████████| 3535/3535 [00:06<00:00, 556.11sección/s]



➡️ Procesando categoría: SECUNDARIA Y BACHILLERATO


...


➡️ Procesando categoría: EDUCACIÓN ESPECIAL


Accesibilidad educación_especial: 100%|██████████████████████████████████████| 3535/3535 [00:05<00:00, 639.25sección/s]



➡️ Procesando categoría: FORMACIÓN PROFESIONAL


Accesibilidad formación_profesional: 100%|███████████████████████████████████| 3535/3535 [00:05<00:00, 612.21sección/s]


,CUSEC,dist_min_infantil_y_primaria_km,n_infantil_y_primaria_1km,n_infantil_y_primaria_5km,n_infantil_y_primaria_15km,n_infantil_y_primaria_30km,disp_ponderada_infantil_y_primaria,dist_min_adultos_e_idiomas_km,n_adultos_e_idiomas_1km,n_adultos_e_idiomas_5km,...,n_educación_especial_5km,n_educación_especial_15km,n_educación_especial_30km,disp_ponderada_educación_especial,dist_min_formación_profesional_km,n_formación_profesional_1km,n_formación_profesional_5km,n_formación_profesional_15km,n_formación_profesional_30km,disp_ponderada_formación_profesional
0,0500101001,5.688878,0,0,4,16,2.4,15.854158,0,0,...,0,0,0,0.0,27.947975,0,0,0,1,0.1
1,0500201001,0.000000,2,2,11,24,6.0,6.277992,0,0,...,0,0,0,0.0,36.877357,0,0,0,0,0.0
2,0500201002,4.415298,0,1,13,25,5.4,8.489764,0,0,...,0,0,0,0.0,33.990424,0,0,0,0,0.0
3,0500501001,4.689838,0,1,3,35,4.4,25.572883,0,0,...,0,0,1,0.1,23.717483,0,0,0,2,0.2
4,0500701001,7.401490,0,0,5,17,2.7,7.357844,0,0,...,0,0,0,0.0,28.537997,0,0,0,1,0.1


# Export de los csv construidos

In [11]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DA, exist_ok=True)

# Rutas de salida
ruta_resultado = os.path.join(DATA_OUTPUTS_DA, "accesibilidad_centros_docentes.csv")
ruta_centros = os.path.join(DATA_OUTPUTS_DA, "centros_docentes_final.csv")
ruta_relaciones = os.path.join(DATA_OUTPUTS_DA, "relaciones_centros_docentes.csv")

# Guardar DataFrames
df_resultado.to_csv(ruta_resultado, index=False, encoding="utf-8-sig")
df_centros_docentes_relevantes.to_csv(
    ruta_centros,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

df_relaciones_global.to_csv(
    ruta_relaciones,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
print(f"✅ Archivos guardados correctamente en:\n- {DATA_OUTPUTS_DA}")

✅ Archivos guardados correctamente en:
- D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DA_Dim_servicios
